# สาธิต AI ตรวจจับ “คน” ด้วย YOLOv8 (Google Colab – GPU T4)  
**สำหรับนักเรียนระดับมัธยม** (เรียนรู้แบบทีละขั้น: Run โมเดล → ดูผล → Fine-tune → ใช้โมเดลใหม่)

---

## สิ่งที่นักเรียนจะได้เรียนรู้
1. AI มอง “ภาพ/วิดีโอ” แล้วบอกว่าเจอ “คน” อยู่ตรงไหน (กรอบสี่เหลี่ยม Bounding Box)
2. ค่าความมั่นใจ (Confidence) คืออะไร และทำไมต้องตั้ง Threshold
3. การใช้ GPU ช่วยให้ทำงานเร็วขึ้นอย่างไร (Tesla T4)
4. Fine-tune คืออะไร (สอนโมเดลเพิ่มด้วยข้อมูล) และผลลัพธ์หลังฝึกเป็นอย่างไร
5. การใช้งานโมเดลที่ฝึกแล้ว (`best.pt`) เพื่อนำไปตรวจจับซ้ำ

---

## ก่อนเริ่ม (สำคัญมาก)
ไปที่เมนูด้านบนของ Colab:  
**Runtime → Change runtime type → Hardware accelerator: GPU → Save**  
จากนั้นค่อยเริ่มรันเซลล์ตามลำดับจากบนลงล่าง

> หมายเหตุ: Notebook นี้ “แสดงวิดีโอ MP4” ในหน้า Colab อย่างเดียว (ไม่ดาวน์โหลดไฟล์)


## 1) ติดตั้งเครื่องมือที่จำเป็น (Install)
เราจะใช้ไลบรารีชื่อ **Ultralytics** ซึ่งทำให้การใช้ YOLO ง่ายมาก  
- `ultralytics` = ตัวโมเดล YOLO + คำสั่ง train/predict  
- `opencv-python` = จัดการวิดีโอ/ภาพบางส่วน (ช่วยให้ระบบทำงานกับไฟล์วิดีโอได้ดี)

ให้กดรันเซลล์ด้านล่าง 1 ครั้ง และรอจนจบ (อาจใช้เวลา 1–3 นาที)

In [ ]:
!pip -q install ultralytics opencv-python
import ultralytics
ultralytics.checks()


## 2) ตรวจสอบว่าเราได้ใช้ GPU หรือยัง (Check GPU)
GPU ช่วยให้การประมวลผลภาพ/วิดีโอเร็วขึ้นมาก  
เราต้องการเห็นว่า `CUDA available: True` และชื่อ GPU เป็น **Tesla T4** (ส่วนใหญ่ใน Colab)

ถ้า CUDA เป็น False ให้กลับไปตั้งค่า GPU ตามขั้น “ก่อนเริ่ม” อีกครั้ง

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 3) โหลดโมเดล YOLOv8 (Base Model)
เราจะใช้ **yolov8n.pt**  
- ตัว `n` = nano (เล็ก/เร็ว) เหมาะกับการสาธิตบน Colab  
โมเดลนี้ถูกฝึกมาล่วงหน้า (pretrained) บนชุดข้อมูลมาตรฐาน (เช่น COCO)

### ทำไมต้อง `classes=[0]`?
ในชุดข้อมูล COCO:  
- class id `0` หมายถึง **person** (คน)  
ดังนั้นเรากรองให้โมเดลแสดงเฉพาะ “คน” เพื่อให้ตรงกับหัวข้อการสาธิต

In [ ]:
from ultralytics import YOLO
base_model = YOLO('yolov8n.pt')
base_model


## 4) ทดลองตรวจจับ “คน” จากภาพ (Inference on Image)
### ขั้นตอน
1) อัปโหลดภาพ 1 รูป  
2) ให้โมเดลตรวจจับคน  
3) แสดงผลเป็นรูปที่มีกรอบสี่เหลี่ยม

### คำศัพท์สำคัญ
- **Bounding box**: กรอบสี่เหลี่ยมรอบวัตถุที่เจอ  
- **Confidence**: ความมั่นใจของโมเดล (0 ถึง 1)  
- `conf=0.25`: หมายถึง “ถ้าความมั่นใจต่ำกว่า 0.25 ไม่ต้องแสดงผล”

In [ ]:
from google.colab import files
uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
print('Image:', img_path)


In [ ]:
# ตรวจจับเฉพาะคน (person) ด้วย GPU (device=0)
base_img_results = base_model.predict(
    source=img_path,
    classes=[0],
    conf=0.25,
    device=0,
    save=True
)

# แสดงรูปผลลัพธ์ใน Colab
import matplotlib.pyplot as plt
base_img_plot = base_img_results[0].plot()
plt.figure(figsize=(10,6))
plt.title('ผลลัพธ์: Base model (yolov8n.pt)')
plt.imshow(base_img_plot)
plt.axis('off')
plt.show()


## 5) ทดลองตรวจจับ “คน” จากวิดีโอ (Inference on Video) + แปลงเป็น MP4 เพื่อดูใน Colab
บางครั้งไฟล์ output อาจเป็น `.avi` ซึ่งเปิดดูบนหน้าเว็บได้ยาก  
เราจึงแปลงเป็น **MP4** ด้วย `ffmpeg` แล้วแสดงใน Colab

### ขั้นตอน
1) อัปโหลดวิดีโอสั้น ๆ (10–30 วินาทีจะดีที่สุด)  
2) โมเดลตรวจจับคนในวิดีโอ  
3) หาไฟล์ผลลัพธ์ล่าสุดในโฟลเดอร์ `runs/detect/`  
4) แปลงเป็น `base_output.mp4` แล้วแสดงผล

In [ ]:
from google.colab import files
uploaded = files.upload()
vid_path = next(iter(uploaded.keys()))
print('Video:', vid_path)


In [ ]:
# ตรวจจับคนในวิดีโอ
base_vid_results = base_model.predict(
    source=vid_path,
    classes=[0],
    conf=0.25,
    device=0,
    save=True
)

# หาไฟล์วิดีโอ output ล่าสุด
import glob, os
from IPython.display import Video, display

cands = sorted(glob.glob('runs/detect/predict*/**/*.*', recursive=True))
base_video_raw = None
for p in reversed(cands):
    if os.path.splitext(p)[1].lower() in ['.mp4', '.avi', '.mov', '.mkv']:
        base_video_raw = p
        break
print('Raw output:', base_video_raw)

# แปลงเป็น MP4 เพื่อดูในหน้า Colab
base_video_mp4 = 'base_output.mp4'
!ffmpeg -y -i "{base_video_raw}" -vcodec libx264 -acodec aac {base_video_mp4}

display(Video(base_video_mp4, embed=True))


# ส่วนสำคัญของบทเรียน: Fine-tune (สอนโมเดลเพิ่ม)

## 6) Fine-tune คืออะไร?
**Fine-tune** หมายถึง “นำโมเดลที่เก่งอยู่แล้ว (pretrained) มาเรียนรู้เพิ่ม”  
เพื่อให้เหมาะกับงาน/ข้อมูลของเรา

### วันนี้เราจะสาธิตด้วย public dataset ชื่อ COCO8
- COCO8 เป็นชุดข้อมูลตัวอย่างขนาดเล็ก (เหมาะกับเดโม เพราะเร็ว)  
- ในการฝึก เราจะเห็นค่าต่าง ๆ เช่น **loss** และผลประเมิน

> หมายเหตุ: COCO8 มีหลายคลาส แต่เราจะยังคง “แสดงเฉพาะคน” ตอนตรวจจับด้วย `classes=[0]` เหมือนเดิม

In [ ]:
# ดาวน์โหลดและแตกไฟล์ COCO8 (ใช้ -o เพื่อไม่ให้ค้างถาม replace)
from ultralytics.utils.downloads import download
download('https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip')
!unzip -o -q coco8.zip
!ls


## 6.1) เริ่มฝึกโมเดล (Train)
เราจะฝึกแบบสั้น ๆ สำหรับเดโม เช่น **epochs=3**  
- **epoch** = รอบที่โมเดลเรียนรู้ผ่านข้อมูลทั้งหมด 1 ครั้ง  
ยิ่ง epochs มาก อาจเก่งขึ้น แต่ใช้เวลามากขึ้น และอาจ overfit ได้

ให้รันเซลล์นี้ แล้วรอดูผล (loss/metrics)

In [ ]:
from ultralytics import YOLO
ft_model = YOLO('yolov8n.pt')
ft_model.train(
    data='coco8.yaml',
    epochs=3,
    imgsz=640,
    batch=16,
    device=0
)


## 6.2) หาไฟล์โมเดลที่ฝึกเสร็จแล้ว (best.pt)
เมื่อฝึกเสร็จ Ultralytics จะสร้างไฟล์น้ำหนักโมเดลใหม่ เช่น `best.pt`  
ไฟล์นี้คือ “โมเดลที่เราฝึกเพิ่มแล้ว” และเราจะนำไปใช้ตรวจจับต่อทันที

In [ ]:
import glob
best_candidates = sorted(glob.glob('runs/**/weights/best.pt', recursive=True))
print('Found best.pt candidates:')
for p in best_candidates[-5:]:
    print(' -', p)

best_pt = best_candidates[-1] if best_candidates else None
print('\nUsing best.pt:', best_pt)


## 7) ใช้โมเดลที่ฝึกแล้ว (After Fine-tune) ตรวจจับ “ภาพเดิม” เพื่อเปรียบเทียบ
เราจะทำ 2 อย่าง:
1) โหลดโมเดลที่ฝึกแล้ว (`best.pt`)  
2) ตรวจจับ “ภาพเดิม” แล้วเอามาเทียบกับ Base model

เป้าหมายคือให้นักเรียนเห็นว่า “โมเดลที่ฝึกแล้ว” ถูกนำไปใช้งานต่อได้จริง

In [ ]:
assert best_pt is not None, 'ไม่พบ best.pt — กรุณาตรวจว่า train สำเร็จหรือไม่'
tuned_model = YOLO(best_pt)
tuned_model


In [ ]:
# ตรวจจับภาพเดิมด้วยโมเดลที่ฝึกแล้ว
tuned_img_results = tuned_model.predict(
    source=img_path,
    classes=[0],
    conf=0.25,
    device=0,
    save=True
)

import matplotlib.pyplot as plt
tuned_img_plot = tuned_img_results[0].plot()
plt.figure(figsize=(10,6))
plt.title('ผลลัพธ์: After Fine-tune (best.pt)')
plt.imshow(tuned_img_plot)
plt.axis('off')
plt.show()


### 7.1) เปรียบเทียบรูป “ก่อน–หลัง” แบบชัด ๆ
ซ้าย: Base model (ก่อนฝึกเพิ่ม)  
ขวา: โมเดลหลัง Fine-tune (best.pt)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))
plt.subplot(1,2,1)
plt.title('ก่อน: Base (yolov8n.pt)')
plt.imshow(base_img_plot)
plt.axis('off')

plt.subplot(1,2,2)
plt.title('หลัง: Tuned (best.pt)')
plt.imshow(tuned_img_plot)
plt.axis('off')
plt.show()


## 8) ใช้โมเดลที่ฝึกแล้ว (After Fine-tune) ตรวจจับ “วิดีโอเดิม” + แปลงเป็น MP4
เหมือนข้อ 5 แต่เปลี่ยนโมเดลเป็น `best.pt`  
แล้วแสดง `tuned_output.mp4` ในหน้า Colab

In [ ]:
# ตรวจจับวิดีโอเดิมด้วยโมเดลที่ฝึกแล้ว
tuned_vid_results = tuned_model.predict(
    source=vid_path,
    classes=[0],
    conf=0.25,
    device=0,
    save=True
)

import glob, os
from IPython.display import Video, display

cands = sorted(glob.glob('runs/detect/predict*/**/*.*', recursive=True))
tuned_video_raw = None
for p in reversed(cands):
    if os.path.splitext(p)[1].lower() in ['.mp4', '.avi', '.mov', '.mkv']:
        tuned_video_raw = p
        break
print('Raw output:', tuned_video_raw)

tuned_video_mp4 = 'tuned_output.mp4'
!ffmpeg -y -i "{tuned_video_raw}" -vcodec libx264 -acodec aac {tuned_video_mp4}
display(Video(tuned_video_mp4, embed=True))


# สรุปบทเรียน (สำหรับครู/ผู้ช่วยวิทยากร)
- นักเรียนได้เห็น **AI ตรวจจับคน** จากภาพและวิดีโอ  
- เข้าใจแนวคิด **Confidence threshold** และการใช้ GPU  
- เห็นกระบวนการ **Fine-tune** และการนำโมเดลที่ฝึกแล้ว (`best.pt`) ไปใช้งานต่อจริง  
- ได้เปรียบเทียบผล “ก่อน–หลัง” อย่างเป็นรูปธรรม

## คำถามชวนคิด (ถามนักเรียน)
1) ถ้า `conf` ตั้งสูงเกินไป จะเกิดอะไรขึ้น?  
2) ถ้า `conf` ตั้งต่ำเกินไป จะเกิดอะไรขึ้น?  
3) ทำไมเราต้องใช้ dataset ที่ “ใกล้เคียง” งานจริงของเราเวลาจะ Fine-tune?  
